### Filter and Clean Metadata

In [23]:
import pandas as pd
metadata = pd.read_csv('./99_Original_Data/metadata_for_study_final_250502.csv', sep = ',')
# Filter out unnecessary columns
## remove the columns: "Unnamed: 0", "F/U activity", "whole saliva(cc)"
metadata = metadata.drop(columns=["F/U activity"])
metadata = metadata.drop(columns=["whole saliva(cc)"])
metadata = metadata.drop(columns=["Disease activity"])

In [24]:
# Change column names
metadata.columns = metadata.columns.str.replace('Smoking (Current, Ex, Never) - 양도 같이', 'Smoking_Status')
metadata.columns = metadata.columns.str.replace('antibiotics 복용유무', 'Antibiotics_Use')
metadata.columns = metadata.columns.str.replace('probiotics 복용유무', 'Probiotics_Use')
metadata.columns = metadata.columns.str.replace('Pt number', 'Patient_Number')
metadata.columns = metadata.columns.str.replace('Alcohol (종류, 양)', 'Alcohol_Use')
metadata.columns = metadata.columns.str.replace('DAS-28 ESR', 'DAS_28_ESR')
metadata.columns = metadata.columns.str.replace('DAS-28 CRP', 'DAS_28_CRP')
metadata.columns = metadata.columns.str.replace('TJC (0/28)', 'TJC.0.28')
metadata.columns = metadata.columns.str.replace('SJC (0/28)', 'SJC.0.28')
metadata.columns = metadata.columns.str.replace('Anti-ccp', 'Anti_ccp')
metadata.columns = metadata.columns.str.replace('DAS-28 ESR.1', 'DAS_28_ESR_1')
metadata.columns = metadata.columns.str.replace('DAS-28 CRP.1', 'DAS_28_CRP_1')
metadata.columns = metadata.columns.str.replace('Good Rx', 'Good_Rx')


In [25]:
# Change the values for some columns
values = list(metadata['Steroid'].unique())
for value in values:
    if value == 'Solondo ':
        metadata['Steroid'] = metadata['Steroid'].replace({value: 'Solondo'})
values = list(metadata['Smoking_Status'].unique())
for value in values:
    if value.startswith('Current'):
        metadata['Smoking_Status'] = metadata['Smoking_Status'].replace({value: 'Current_Smoker'})
    elif value.startswith('Ex'):
        metadata['Smoking_Status'] = metadata['Smoking_Status'].replace({value: 'Former_Smoker'})
    elif value.startswith('Never'):
        metadata['Smoking_Status'] = metadata['Smoking_Status'].replace({value: 'Never_Smoker'})
metadata['Alcohol_Use'] = metadata['Alcohol_Use'].fillna('No_Alcohol')
values = list(metadata['Alcohol_Use'].unique())
for value in values:
    if value == "No_Alcohol" :
        metadata['Alcohol_Use'] = metadata['Alcohol_Use'].replace({value: 'No_Alcohol'})
    else:
        metadata['Alcohol_Use'] = metadata['Alcohol_Use'].replace({value: 'Alcohol_User'})
values = list(metadata['diagnosis'].unique())
# Simplify the diagnosis of RA patients (both SPRA and SNRA to RA)
for value in values:
    if value.endswith('RA'):
        metadata['diagnosis'] = metadata['diagnosis'].replace({value: 'RA'})
    else:
        metadata['diagnosis'] = metadata['diagnosis'].replace({value: 'Non_RA'})
values = list(metadata['NSAIDs'].unique())
for value in values:
    if value == 'no':
        metadata['NSAIDs'] = metadata['NSAIDs'].replace({value: 'No_NSAIDs'})
    else:
        metadata['NSAIDs'] = metadata['NSAIDs'].replace({value: 'NSAIDs_User'})
values = list(metadata['Anti_ccp'].unique())
for value in values:
    value = str(value)
    if value == 'negative':
        metadata['Anti_ccp'] = metadata['Anti_ccp'].replace({value: 0})

In [26]:
# Iterate over all columns in the metadata dataframe
for col in metadata.columns:
    # Check if the column is of object type (string-like)
    if metadata[col].dtype == 'object':
        # Replace values containing '>' or '<' by removing the symbol
        metadata[col] = metadata[col].astype(str).apply(lambda x: x.lstrip('<').lstrip('>') if x.startswith('<') or x.startswith('>') else x)
        # Try to convert back to numeric if possible
        try:
            metadata[col] = pd.to_numeric(metadata[col], errors='ignore')
        except:
            pass

/tmp/ipykernel_1674395/3521881083.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata[col] = pd.to_numeric(metadata[col], errors='ignore')
/tmp/ipykernel_1674395/3521881083.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata[col] = pd.to_numeric(metadata[col], errors='ignore')
/tmp/ipykernel_1674395/3521881083.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  metadata[col] = pd.to_numeric(metadata[col], errors='ignore')
/tmp/ipykernel_1674395/3521881083.py:9: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  me

In [27]:
# Add "sample_" to name
metadata['saliva swab'] = "sample_" + metadata['saliva swab'].astype(str)
metadata['saliva swab'] = metadata['saliva swab'].str.replace("sample_2022120210101", "sample_10810599")
metadata['saliva swab'] = metadata['saliva swab'].str.replace("sample_2022120210102", "sample_5192784")
metadata['saliva swab'] = metadata['saliva swab'].str.replace("sample_2022120210103", "sample_5192779")
metadata['saliva swab'] = metadata['saliva swab'].str.replace("sample_2022120210104", "sample_10816884")

In [28]:
# Save the cleaned metadata to a new file
metadata.to_csv('./01_Processed_Data/cleaned_metadata.tsv', sep='\t', index=True)

In [30]:
# Make file for RA patients with follow-up data for analysis
## Take only samples with follow-up 'DAS-28 ESR.1' value
metadata = metadata[metadata['DAS_28_ESR.1'].notna()]
## make treatment response column using EULAR response criteria standard
def classify_eular_response(row):
    """
    Classifies patient response based on EULAR criteria using 
    DAS28 endpoint score and the Delta (Improvement).
    
    Logic derived from standard EULAR response matrix:
    - Good: Improvement > 1.2 AND Endpoint <= 3.2
    - None: Improvement <= 0.6 OR (Improvement <= 1.2 AND Endpoint > 5.1)
    - Moderate: All other cases
    """
    
    # Extract values for cleaner reading
    endpoint = row['DAS_28_ESR.1']
    improvement = row['Delta DAS_28_ESR'] # Assumes (Baseline - FollowUp)
    
    # 1. Check for NO RESPONSE cases first
    # Criterion A: Improvement is 0.6 or less (regardless of endpoint score)
    if improvement <= 0.6:
        return 'None'
    
    # Criterion B: Endpoint is high (>5.1) and improvement is small (between 0.6 and 1.2)
    # Note: If improvement was > 1.2 with score > 5.1, it would be Moderate.
    if improvement <= 1.2 and endpoint > 5.1:
        return 'None'

    # 2. Check for GOOD RESPONSE cases
    # Criterion: Low disease activity (<= 3.2) AND high improvement (> 1.2)
    if improvement > 1.2 and endpoint <= 3.2:
        return 'Good'

    # 3. Everything else is MODERATE
    # This covers:
    # - Endpoint <= 3.2 but Improvement is 0.6-1.2
    # - Endpoint 3.2-5.1 and Improvement > 0.6
    # - Endpoint > 5.1 and Improvement > 1.2
    return 'Moderate'

# 2. Apply the classification function
# axis=1 tells pandas to apply the function row-by-row
metadata['EULAR_Response'] = metadata.apply(classify_eular_response, axis=1)

# Consider only Good responder as Responder and Non-good responders as Non-Responder
metadata['Responder_Status'] = metadata['EULAR_Response'].apply(lambda x: 'Responder' if x == 'Good' else 'Non-Responder')

In [31]:
# Save the metadata file for RA patients analysis
metadata.to_csv('./01_Processed_Data/cleaned_metadata_RA.tsv', sep='\t', index=False)